# 180 — Attention Pattern Clustering

Cluster attention patterns to discover archetypes, measure head similarity,
track pattern diversity, test stability across inputs, and trace evolution.

## Why This Matters

Attention pattern clustering reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_pattern_clustering import (
    pattern_archetypes,
    head_pattern_similarity,
    pattern_diversity,
    pattern_stability_across_inputs,
    cross_layer_pattern_evolution,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.arange(15)

In [ ]:
result = pattern_archetypes(model, tokens, n_clusters=4)
print(f"Found {result['n_clusters']} archetypes from {result['n_total_heads']} heads:\n")
for c in result['clusters']:
    members = ', '.join(f'L{m["layer"]}H{m["head"]}' for m in c['members'])
    print(f"Cluster {c['cluster']} ({c['n_members']} heads): entropy={c['mean_entropy']:.3f}  "
          f"diag={c['diag_score']:.3f}  prev={c['prev_token_score']:.3f}")
    print(f"  Members: {members}\n")

In [ ]:
result = head_pattern_similarity(model, tokens, layer=0)
print("Within-layer pattern similarity:\n")
for row in result['similarity_matrix']:
    vals = '  '.join(f'{v:.3f}' for v in row)
    print(f"  [{vals}]")
print(f"\nMost similar pair: {result['most_similar'][0]}")

In [ ]:
result = pattern_diversity(model, tokens)
for layer in result['per_layer']:
    bar = '#' * int(layer['diversity'] * 30)
    print(f"Layer {layer['layer']}: diversity={layer['diversity']:.3f}  "
          f"sim={layer['mean_pairwise_similarity']:.3f}  {bar}")

In [ ]:
tokens2 = jnp.arange(20, 35)
result = pattern_stability_across_inputs(model, tokens, tokens2)
print(f"Stable heads: {result['n_stable']}/{result['n_total']} "
      f"({result['stability_fraction']:.1%})\n")
for h in result['per_head']:
    status = 'STABLE' if h['is_stable'] else 'varies'
    print(f"  L{h['layer']}H{h['head']}: cos={h['cosine_similarity']:.3f}  {status}")

In [ ]:
result = cross_layer_pattern_evolution(model, tokens)
print(f"Most evolving head index: {result['most_evolving_head']}\n")
for h in result['per_head']:
    transitions = ', '.join(f'L{t["from_layer"]}->L{t["to_layer"]}({t["cosine"]:.3f})'
                           for t in h['layer_transitions'])
    print(f"Head {h['head']}: mean_change={h['mean_change']:.3f}  [{transitions}]")